擷取網址正文 → 切成 chunks → 寫入 Weaviate → 依問題搜尋 chunks → 由LLM產生答案與引用。

In [39]:
import os
from datetime import datetime, timezone
from ipaddress import ip_address
from pathlib import Path
from socket import gethostbyname
from urllib.parse import urlparse
from uuid import uuid4

import requests
import weaviate
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from weaviate.classes.config import Configure, DataType, Property
from weaviate.classes.query import Filter, MetadataQuery

load_dotenv(override=True)

COLLECTION = "FeedbackLoopDocuments"
OLLAMA_BASE_URL = "http://120.113.70.238:11434"
EMBEDDING_MODEL = "nomic-embed-text:latest"
LLM_MODEL = "gemma4:26b"
CHUNK_SIZE = 700
CHUNK_OVERLAP = 120
TOP_K = 5

print(f"Ollama: {OLLAMA_BASE_URL}")
print(f"LLM: {LLM_MODEL}")
print(f"Embedding: {EMBEDDING_MODEL}")

Ollama: http://120.113.70.238:11434
LLM: gemma4:26b
Embedding: nomic-embed-text:latest


##### 驗證網址、擷取正文並切成 chunks
先拒絕內網與保留 IP，接著移除 script、style、nav、footer 等雜訊。`chunk_text` 會保留重疊文字，避免答案剛好落在兩段交界處。

In [40]:
def validate_public_url(url):
    parsed = urlparse(url)
    if parsed.scheme not in {"http", "https"} or not parsed.hostname:
        raise ValueError("URL 必須是 http 或 https 網址。")
    address = ip_address(gethostbyname(parsed.hostname))
    if address.is_private or address.is_loopback or address.is_link_local or address.is_reserved:
        raise ValueError("不允許存取內網或保留 IP。")
    return url

def load_web_page(url):
    validate_public_url(url)
    response = requests.get(url, timeout=15, headers={"User-Agent": "FeedbackLoop-AI/0.1"})
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer"]):
        tag.decompose()
    content = soup.get_text(" ", strip=True)
    if not content:
        raise ValueError("網頁沒有可用的正文內容。")
    title = soup.title.get_text(strip=True) if soup.title else url
    return {"url": url, "title": title, "content": content}

def chunk_text(content, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = " ".join(content.split())
    chunks, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            boundary = text.rfind(" ", start, end)
            if boundary > start:
                end = boundary
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = max(end - overlap, start + 1)
        while start < len(text) and text[start].isspace():
            start += 1
    return chunks

def embedding(text):
    response = requests.post(f"{OLLAMA_BASE_URL}/api/embeddings",json={"model": EMBEDDING_MODEL, "prompt": text},timeout=30,)
    response.raise_for_status()
    vector = response.json().get("embedding")
    if not isinstance(vector, list) or not vector:
        raise ValueError("Embedding service returned no vector.")
    return vector

##### 資料庫連線與 collection
`connect_database` 連到本機 Weaviate；`get_collection` 建立或補齊 `FeedbackLoopDocuments` 欄位。網站 chunk 會保存 URL、標題、chunk 編號與向量。

In [41]:
WEB_PROPERTIES = [
    Property(name="chunk_id", data_type=DataType.TEXT),
    Property(name="document_id", data_type=DataType.TEXT),
    Property(name="user_id", data_type=DataType.TEXT),
    Property(name="source_type", data_type=DataType.TEXT),
    Property(name="filename", data_type=DataType.TEXT),
    Property(name="url", data_type=DataType.TEXT),
    Property(name="title", data_type=DataType.TEXT),
    Property(name="chunk_index", data_type=DataType.INT),
    Property(name="content", data_type=DataType.TEXT),
    Property(name="created_at", data_type=DataType.DATE),
]

def connect_database():
    return weaviate.connect_to_local(
        host=os.getenv("WEAVIATE_HOST", "127.0.0.1"),
        port=int(os.getenv("WEAVIATE_PORT", "8080")),
        grpc_port=int(os.getenv("WEAVIATE_GRPC_PORT", "50051")),
    )

def get_collection(client):
    if not client.collections.exists(COLLECTION):
        client.collections.create(COLLECTION, vector_config=Configure.Vectors.self_provided(), properties=WEB_PROPERTIES)
    collection = client.collections.get(COLLECTION)
    existing = {prop.name for prop in collection.config.get().properties}
    for prop in WEB_PROPERTIES:
        if prop.name not in existing:
            collection.config.add_property(prop)
    return collection

client = connect_database()
try:
    print("Weaviate ready:", client.is_ready())
    print("Weaviate live:", client.is_live())
finally:
    client.close()

Weaviate ready: True
Weaviate live: True


##### 網址內容切成 chunks 並寫入資料庫
建立 `document_id`，並將每個 chunk 的 metadata 與 embedding 寫入 Weaviate。`document_id` 是後續搜尋答案的範圍

In [42]:
def ingest_web_url(url):
    page = load_web_page(url)
    document_id = uuid4().hex
    chunks = chunk_text(page["content"])
    client = connect_database()
    try:
        collection = get_collection(client)
        for index, content in enumerate(chunks, start=1):
            properties = {
                "chunk_id": uuid4().hex,
                "document_id": document_id,
                "user_id": "local",
                "source_type": "web",
                "filename": page["title"],
                "url": page["url"],
                "title": page["title"],
                "chunk_index": index,
                "content": content,
                "created_at": datetime.now(timezone.utc).isoformat(),
            }
            collection.data.insert(properties=properties, vector=embedding(content))
    finally:
        client.close()
    return document_id, page, len(chunks)

TEST_URL = "https://zh.wikipedia.org/zh-tw/%E9%99%BD%E5%B2%B1%E9%8B%BC"
document_id, page, chunk_count = ingest_web_url(TEST_URL)
print(f"已寫入 {chunk_count} 個 chunks")
print(f"document_id: {document_id}")
print(f"title: {page['title']}")

已寫入 22 個 chunks
document_id: 766c8662bcb3482194d7dec713c1288c
title: 陽岱鋼 - 維基百科，自由的百科全書


##### 如何從資料庫找答案
將問題轉成 embedding，再以 `near_vector` 搜尋最相近的 Top-K chunks；`document_id` filter 確保不會混入其他網址的內容。

In [43]:
def retrieve_chunks(question, document_id):
    client = connect_database()
    try:
        collection = get_collection(client)
        response = collection.query.near_vector(
            near_vector=embedding(question),
            filters=Filter.by_property("document_id").equal(document_id),
            limit=TOP_K,
            return_metadata=MetadataQuery(distance=True),
        )
        return [{**item.properties, "score": 1 - (item.metadata.distance or 0)} for item in response.objects]
    finally:
        client.close()

QUESTION = "請根據本文整理重點。"
retrieved = retrieve_chunks(QUESTION, document_id)
assert retrieved, "沒有找到相關 chunks，請確認 document_id 與資料庫連線。"

for chunk in retrieved:
    print(f"chunk {chunk['chunk_index']} | score={chunk['score']:.3f}")
    print(chunk['content'][:180], "...\n")

chunk 10 | score=0.609
021年） 參考文獻 [ 編輯 ] ^ 102年精英獎終生成就獎及傑出獎(入圍者)名單出爐！ . 中央社 CNA. 2013-11-28 [ 2019-11-27 ] . （原始內容 存檔 於2019-06-18） （中文（臺灣）） . ^ https://web.archive.org/web/20190331040254/https://vk.sport ...

chunk 12 | score=0.603
022-10-20 ] . （原始內容 存檔 於2022-10-20） （中文（臺灣）） . ^ 湖國碼頭犬隊合約到期！陽岱鋼在台灣自主訓練 下一步掀關注 . 民視. 2023-02-15 [ 2023-02-15 ] . （原始內容 存檔 於2023-05-01）. ^ 本季落腳獨盟北卡球隊陽岱鋼：想見識神的國度 . TSNA. 2023-03-09 [  ...

chunk 13 | score=0.602
一軍？ . NOWnews 今日新聞. 2024-11-14 [ 2024-11-14 ] . （原始內容 存檔 於2024-11-14）. ^ 謝靜雯. 龍柏安 , 編. 陽岱鋼仍想拚回日職一軍 戰力外測試4打席1保送 . 台北. 中央社 CNA. 2024-11-14 [ 2024-11-14 ] . （原始內容 存檔 於2024-11-14） （中文（ ...

chunk 1 | score=0.584
陽岱鋼 - 維基百科，自由的百科全書 跳至內容 搜尋 搜尋 陽岱鋼 4種語言 مصرى English 日本語 한국어 編輯連結 維基百科，自由的百科全書 本條目存在以下問題 ，請協助 改善本條目 或在 討論頁 針對議題發表看法。 此 生者傳記 條目需要補充更多 可供查證 的 來源 。 ( 2021年2月20日 ) 請協助補充 可靠來源 ，無法查證的在世人物 ...

chunk 22 | score=0.564
台灣男子棒球運動員 台灣原住民棒球運動員 北海道日本火腿鬥士球員 讀賣巨人球員 台灣旅日本棒球運動員 阿美族人 臺東市人 陽姓 馬蘭陽家 布里斯本俠盜球員 2006年世界棒球經典賽中華台北代表隊選手 2006年洲際盃棒球賽中華台北代表隊選手 2006年亞運棒球賽中華台北代表隊選手 2

##### 用檢索結果產生答案與檢查精準度
剛剛檢索出的 chunks。將 `EXPECTED_ANSWER_FRAGMENT` 改成從本文驗證的關鍵事實；Notebook 會檢查回答是否含有該片段，並印出引用來源。

In [44]:
EXPECTED_ANSWER_FRAGMENT = ""  # 例如："2026"；留白則只顯示答案與引用。

def generate_answer(question, chunks):
    contexts = "\n\n".join(
        f"[{chunk['title']} | {chunk['url']}]\n{chunk['content']}"
        for chunk in chunks
    )
    prompt = (
        "只根據下列來源回答。資料不足時請明確回答資料不足；不可捏造來源以外的內容。"
        f"\n\n問題：{question}\n\n來源：\n{contexts}"
    )
    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json={"model": LLM_MODEL, "prompt": prompt, "stream": False},
        timeout=120,
    )
    response.raise_for_status()
    return response.json()["response"].strip()

answer = generate_answer(QUESTION, retrieved)
print(answer)

print("\n引用來源：")
for chunk in retrieved:
    print(f"- {chunk['title']} | {chunk['url']} | chunk {chunk['chunk_index']} | score={chunk['score']:.3f}")

if EXPECTED_ANSWER_FRAGMENT:
    assert EXPECTED_ANSWER_FRAGMENT in answer, "答案未包含預期片段。"
    print("\n精準度檢查：通過")

根據您提供的來源，關於陽岱鋼的重點整理如下：

**1. 基本資料**
* **出生資訊**：1987 年 1 月 17 日（39 歲），出生於台灣臺東縣臺東市。
* **族裔與身分**：阿美族人。
* **球技特性**：右投、右打；位置為外野手或一壘手。

**2. 職業生涯與效力球隊**
* **日本職棒 (NPB)**：
    * 北海道日本火腿鬥士（2006 年－2016 年）。
    * 讀賣巨人（2017 年－2021 年）。
* **其他聯盟與球隊**：
    * 湖國碼頭犬 (Lake Country DockHounds)（2 年）。
    * 高點搖椅 (High Point Rockers)（2023 年）。
    * Oisix 新潟天鵝之皇（2024 年－）。
* **國家隊紀錄**：曾代表中華台北參加 2006 年世界棒球經典賽、2006 年洲際盃、2006 年亞運、2007 年亞洲棒球錦標賽及 2015 年世界棒球 12 強賽。

**3. 成就與紀錄**
* **日本職棒數據（截至 2021 年球季）**：打擊率 0.270、105 支全壘打、482 分打點、141 次盜壘。
* **主要榮譽**：
    * 2016 年日本大賽冠軍。
    * 2013 年太平洋聯盟盜壘王。
    * 2012 年、2013 年、2014 年及 2016 年太平洋聯盟外野手金手套獎。
    * 2013 年世界棒球經典賽 MVP。

**4. 近期動態**
* **加盟情況**：2024 年加盟日職 2 軍新潟天鵝皇隊，背號為 1 號。
* **近期表現與目標**：2024 年 11 月參加戰力外測試，成績為 4 打席 1 保送；他表示仍強烈渴望重返日職一軍。

引用來源：
- 陽岱鋼 - 維基百科，自由的百科全書 | https://zh.wikipedia.org/zh-tw/%E9%99%BD%E5%B2%B1%E9%8B%BC | chunk 10 | score=0.609
- 陽岱鋼 - 維基百科，自由的百科全書 | https://zh.wikipedia.org/zh-tw/%E9%99%BD%E5%B2%B1%E9%8B%BC | chunk 12 | score=0.603
- 陽岱鋼 - 維基百科，自由的百科